In [ ]:
import os

import pandas as pd

from scGPT import load_scgpt, extract_model_weights, SCGPT_DEFS

from napistu.utils import download_wget
from napistu.constants import ONTOLOGIES

from utils import load_model_weights, compute_attention_from_weights

DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scGPT_bc")
MODEL_RESULTS_PATH = os.path.join(OUTPUT_DIR, "scgpt_weights.npz")
ANNOTATIONS_PATH = os.path.join(DATA_DIR, "scgpt_gene_info.csv")

/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/opt/homebrew/Caskroom/miniforge/base/envs/scgpt/lib/python3.11/site-packages/scg

In [2]:
# download ensembl <-> aliases mappings
if not os.path.isfile(ANNOTATIONS_PATH):
    download_wget(SCGPT_DEFS.GENE_IDENTIFIERS_URL, ANNOTATIONS_PATH)

gene_annotations = (
    pd.read_csv(ANNOTATIONS_PATH, index_col = 0)
    .rename(columns = {
        "feature_id" : ONTOLOGIES.ENSEMBL_GENE
    })
)

In [4]:
model, vocab, model_metadata = load_scgpt(MODEL_PATH)
extract_model_weights(model, vocab, MODEL_RESULTS_PATH)

Resume model from scGPT_bc/best_model.pt, the model args will override the config scGPT_bc/args.json.
Loading params encoder.embedding.weight with shape torch.Size([60697, 512])
Loading params encoder.enc_norm.weight with shape torch.Size([512])
Loading params encoder.enc_norm.bias with shape torch.Size([512])
Loading params value_encoder.linear1.weight with shape torch.Size([512, 1])
Loading params value_encoder.linear1.bias with shape torch.Size([512])
Loading params value_encoder.linear2.weight with shape torch.Size([512, 512])
Loading params value_encoder.linear2.bias with shape torch.Size([512])
Loading params value_encoder.norm.weight with shape torch.Size([512])
Loading params value_encoder.norm.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.weight with shape torch.Size([512, 512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0.linear1.w

In [6]:
# Load any model
scgpt = load_model_weights(MODEL_RESULTS_PATH)

GENES_OF_INTEREST = gene_annotations["feature_name"].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in scgpt['genes']]

# Compute attention on demand
layer_11_attn = compute_attention_from_weights(
    scgpt['embeddings'][GENE_MASK,:],
    scgpt['attention_weights']['layer_11']['W_q'],
    scgpt['attention_weights']['layer_11']['W_k']
)

In [9]:
scgpt["embeddings"].shape

(60697, 512)

In [7]:
layer_11_attn.shape

(20000, 20000)